# Soccer data exploration and cleaning

### Introduction:
In this notebook, I will explore and clean the `soccerdata` dataset!

#### About the dataset:
The dataset is created using the [soccerdata library](https://pypi.org/project/soccerdata/), which is a python library for scraping different types of football data from various sources such as **Fbref, ESPN, Understat, WhoScored.** More specifically, this notebook will analyze and process data from [Fbref](https://fbref.com/en/).

#### About the data scraping:
The project fully complies with all requirements from Fbref and guarantees data security!

#### Why this data:
As the aim of this project is to create sufficiently reliable and accurate enough prediction of football matches, I should consider a lot of different data types.The way this dataset will contribute to the project and its purpose is that it will provide some very important and essential features such as:
- `Match Attendance`: This is more of a external factor, but eventually, it could be used!
- `Match exact time`: All of the other datasets does not provide the exact time at which the matches has been played, or even if they do, it is very limited.However, the importantce of this feature is that it will be combined with the **location of the matches**, as the stadiums in which they have been played, and this way I can get the **weather** which was during the match (at least when it started).This again is more of a external factor, but very useful if used correctly.For example the teams tends to score less goals when the weather is bad.
- `Ball Possesion`: This, again, is a feature which is missing in the other datasets, but it is something very important because it shows which teams has been controlling the game and eventually the possible winner of the match
- `Teams formations`: The team **formations** is maybe one of the most important tactics in football which reveals what how a team would play during a match: more defensive, attacking or more balanced maybe!Many of the teams actions during the matches are strongly based on the formation and the tactic which they have!

#### Foreknowledge of the data:
- As I have observed the structure of the data and the availability of its features, I want to specify that for `season` 2014/2015 of the La Liga matches, there is missing data in the teams **Formations**, and the data is totally missing.So, in order to not leave this season with so much important missing data, in the final data preparation, the missing data will be replaced witj the one from the `Transfermarkt` dataset which provides the exact data which is needed in full availability!

### Now lets get started:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import re
import unicodedata
from pathlib import Path

from football_betting_analysis.config import SOCCER_DATA_INTERIM_PATH, START_SEASON, END_SEASON

from football_betting_analysis.data.text_cleaning import clean_text_values
from football_betting_analysis.data.save_data_into_file import save_data
from football_betting_analysis.data.data_cleaning import convert_string_to_datetime, optimize_dataframe_memory, validate_and_cast_dataframe_dtypes
from football_betting_analysis.data.team_match_validation import validate_team_matches
from football_betting_analysis.data.teams_names_mapper import build_team_mapping

In [2]:
matches_df = pd.read_csv('../../data/raw/soccer_data/matches.csv')

In [3]:
matches_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8360 entries, 0 to 8359
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   league         8360 non-null   str    
 1   season         8360 non-null   int64  
 2   team           8360 non-null   str    
 3   game           8360 non-null   str    
 4   date           8360 non-null   str    
 5   time           8360 non-null   str    
 6   round          8360 non-null   str    
 7   day            8360 non-null   str    
 8   venue          8360 non-null   str    
 9   result         8360 non-null   str    
 10  GF             8360 non-null   int64  
 11  GA             8360 non-null   int64  
 12  opponent       8360 non-null   str    
 13  Poss           8358 non-null   float64
 14  Attendance     7384 non-null   float64
 15  Captain        7600 non-null   str    
 16  Formation      7600 non-null   str    
 17  Opp Formation  7600 non-null   str    
 18  Referee        8360

In [4]:
matches_df.columns

Index(['league', 'season', 'team', 'game', 'date', 'time', 'round', 'day',
       'venue', 'result', 'GF', 'GA', 'opponent', 'Poss', 'Attendance',
       'Captain', 'Formation', 'Opp Formation', 'Referee', 'match_report',
       'Notes'],
      dtype='str')

As I have speciifed there is an entire season without the teams formations data!However, there is also missing data into the Possesion and Attandance features.I will do the exact same thing as I have specified in the introduction!I will just get the data from another dataset, probably from `Transfermarkt`.

In [5]:
matches_df.shape

(8360, 21)

### Data cleaning:
I will clean the data, as much as I could by following a standard process such as:
- fixing the data types
- fixing any text or encoding probelms with the teams names
- mapping the teams names to the ones from the other datasets(**Understat**) - to maintain consistency!
- optimizing the datasets memory usage, as this means that some of the data types can be downcasted to more small-memory ones
- Making all kinds of validations to ensure that the dataset meets all of the minimal requirements for the needs of the project!

#### Fixing the data types:
Now lets begin by fixing the data types of the features.

In [6]:
matches_df.dtypes

league               str
season             int64
team                 str
game                 str
date                 str
time                 str
round                str
day                  str
venue                str
result               str
GF                 int64
GA                 int64
opponent             str
Poss             float64
Attendance       float64
Captain              str
Formation            str
Opp Formation        str
Referee              str
match_report         str
Notes            float64
dtype: object

In [7]:
matches_interim = matches_df.copy()

In [8]:
matches_interim['date'] = convert_string_to_datetime(matches_interim['date'], format_string='%Y-%m-%d')

The date feature is already a datetime object, not lets fix the other features types using the `validate_and_cast_dataframe_dtypes`, which will try to parse the different features to different data types as this will find the one which is more suitable and correct for the specific feature:

In [9]:
matches_interim = validate_and_cast_dataframe_dtypes(matches_interim)

In [10]:
matches_interim.dtypes

league                      str
season                    int64
team                        str
game                        str
date             datetime64[us]
time                        str
round                       str
day                         str
venue                       str
result                      str
GF                        int64
GA                        int64
opponent                    str
Poss                      Int64
Attendance                Int64
Captain                     str
Formation                   str
Opp Formation               str
Referee                     str
match_report                str
Notes                   float64
dtype: object

### Text cleaning:
Cleaning text features, as ensuring that all of the text content in the dataset are with valid encoding, no problematic characters and are all valid:

In [11]:
str_cols = matches_interim.select_dtypes(include=['str']).columns

matches_interim[str_cols] = matches_interim[str_cols].apply(
    clean_text_values
)

### Changing the format of the seasons:
Currently the seasons are represented as a pair of two digits numbers such as: 1415, which refers to season 2014/2015.In order to keep consistency between the datasets, I should change the format of the seasons to their full two years representation!

Before fixing the season feature I will first need to order the dataset in **ascending** order:

In [12]:
matches_interim = matches_interim.sort_values('date', ascending=True)

Now lets reformat the season feature:

In [13]:
matches_interim['season_new'] = np.where(
    matches_interim['date'].dt.month >= 8,
    matches_interim['date'].dt.year.astype(str) + '/' + (matches_interim['date'].dt.year + 1).astype(str),
    (matches_interim['date'].dt.year - 1).astype(str) + '/' + matches_interim['date'].dt.year.astype(str)
)

Some validation that everything is ok:

In [14]:
valid_seasons = [
    f"{year}/{year+1}" for year in range(START_SEASON, END_SEASON)
]

len(matches_interim[matches_interim['season_new'].isin(valid_seasons)]) == len(matches_interim)

True

In [15]:
matches_interim['season'] = matches_interim['season_new']
matches_interim = matches_interim.drop(columns=['season_new'])

### Checking for dupliates:
Now lets check if there are any duplicates into the dataset!As a unique match identifier I will use the combintaion between: **team, opponent, date, time and venue**, because there could not be more than one match with the same two teams at the same data with the same vanue - **venue** because currently the dataset is splitted into **Home and Away** matches and each match exists two times in the dataset, so the venue is very important to be included into the checking criteria:

In [16]:
matches_interim.duplicated(subset=['team', 'date', 'time', 'venue', 'opponent']).any()

np.False_

### Validating the seasons matches count:

Lets see if there are a total of 760 matches for each season:

In [17]:
matches_interim.groupby('season').size()

season
2014/2015    760
2015/2016    760
2016/2017    760
2017/2018    760
2018/2019    760
2019/2020    760
2020/2021    760
2021/2022    760
2022/2023    760
2023/2024    760
2024/2025    760
dtype: int64

Everything seems to be perfect here!

### Validating the amount of unique teams in each of the seasons
I am expecting that there will be a total of 20 unique teams for each of the seasons, just because this is the predefined count of teams which the Spanish La Liga have!

In [18]:
matches_interim.groupby('season')['team'].nunique()

season
2014/2015    20
2015/2016    20
2016/2017    20
2017/2018    20
2018/2019    20
2019/2020    20
2020/2021    20
2021/2022    20
2022/2023    20
2023/2024    20
2024/2025    20
Name: team, dtype: int64

It seems that there is not problem with the matches and the teams in each of the seasons!

### Validating the matches dates: 
A tyical football season starts in August in one year and ends in May in the next year, so lets see the min and max values for each of the seasons matches dates:

In [19]:
matches_interim.groupby('season')['date'].agg(['min', 'max'])

,min,max
season,,
2014/2015,2014-08-23,2015-05-23
2015/2016,2015-08-21,2016-05-15
2016/2017,2016-08-19,2017-05-21
2017/2018,2017-08-18,2018-05-20
2018/2019,2018-08-17,2019-05-19
2019/2020,2019-08-16,2020-07-19
2020/2021,2020-09-12,2021-05-23
2021/2022,2021-08-13,2022-05-22
2022/2023,2022-08-12,2023-06-04


### Merging the matches:
Currently the dataset contains each of its matches **twice**, by splitting them into **Home Team** matches and  **Away Team** matches.However, in order to proceed from on and share the same format and structure as the other datasets, I am required to restructure the dataset by merging the home and away matches into together!I have 11 seasons of matches, each season with 380 matches total and this makes the total of **4,180** matches.If we look at the other perspective, which is how the dataset is structured currently, each team has 38 matches per season, and there are 20 teams in each season, which is 760 matches per season and for all of the season there are the total of **8,360**.

Now lets restructure the matches. \
First I will split the matches into **Home and Away ones**:

In [20]:
home = matches_interim[matches_interim["venue"].str.lower() == "home"].copy()
away = matches_interim[matches_interim["venue"].str.lower() == "away"].copy()

Now I will just combine them together by using the unique identifier for each match which is: **season, date, time,team, opponent**:

In [21]:
# Merge home and away perspectives of the same match
matches_interim = home.merge(
    away,
    left_on=[
        "league",
        "season",
        "date",
        "time",
        "team",
        "opponent"
    ],
    right_on=[
        "league",
        "season",
        "date",
        "time",
        "opponent",
        "team"
    ],
    suffixes=("_home", "_away"),
    validate='one_to_one'
)

In [22]:
matches_interim = pd.DataFrame({
    "league": matches_interim["league"],
    "season": matches_interim["season"],
    "date": matches_interim["date"],
    "time": matches_interim["time"],
    "h_title": matches_interim["team_home"],
    "a_title": matches_interim["team_away"],
    "h_goals": matches_interim["GF_home"],
    "a_goals": matches_interim["GF_away"],
    "h_poss": matches_interim["Poss_home"],
    "a_poss": matches_interim["Poss_away"],
    "attendance": matches_interim["Attendance_home"],
    "referee": matches_interim["Referee_home"],
    "h_formation": matches_interim["Formation_home"],
    "a_formation": matches_interim["Formation_away"],
    "round": matches_interim["round_home"],
    "day": matches_interim["day_home"]
})

In [23]:
matches_interim.shape

(4180, 16)

In [24]:
matches_interim.info()

<class 'pandas.DataFrame'>
RangeIndex: 4180 entries, 0 to 4179
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   league       4180 non-null   str           
 1   season       4180 non-null   str           
 2   date         4180 non-null   datetime64[us]
 3   time         4180 non-null   str           
 4   h_title      4180 non-null   str           
 5   a_title      4180 non-null   str           
 6   h_goals      4180 non-null   int64         
 7   a_goals      4180 non-null   int64         
 8   h_poss       4179 non-null   Int64         
 9   a_poss       4179 non-null   Int64         
 10  attendance   3692 non-null   Int64         
 11  referee      4180 non-null   str           
 12  h_formation  3800 non-null   str           
 13  a_formation  3800 non-null   str           
 14  round        4180 non-null   str           
 15  day          4180 non-null   str           
dtypes: Int64(3), date

### Mapping teams names:
Something which is very important is to keep the teams names consistent across all of the datasets, so that when the datasets are merged together to not have any problems!

The default teams names which this project uses are the ones from the **Understat** dataset.This is for the reason that the Understat dataset represents the teams names in the most understandable and modern way!

So in order to so, I will first need to get the matches data from understat:

In [25]:
understat_matches = pd.read_parquet('../../data/interim/understat_data/interim_matches_v1.parquet')

In [26]:
understat_all_teams = (
    understat_matches
        .groupby('season')['h_title']
            .unique()
            .to_dict()
)

understat_all_teams

{'2014/2015': <ArrowStringArray>
 [             'Malaga',             'Sevilla',             'Granada',
              'Almeria',               'Eibar',           'Barcelona',
           'Celta Vigo',             'Levante',         'Real Madrid',
       'Rayo Vallecano',              'Getafe',            'Valencia',
              'Cordoba',       'Athletic Club',     'Atletico Madrid',
             'Espanyol',          'Villarreal', 'Deportivo La Coruna',
        'Real Sociedad',               'Elche']
 Length: 20, dtype: str,
 '2015/2016': <ArrowStringArray>
 [             'Malaga', 'Deportivo La Coruna',            'Espanyol',
      'Atletico Madrid',      'Rayo Vallecano',       'Athletic Club',
       'Sporting Gijon',             'Levante',          'Real Betis',
              'Granada',          'Villarreal',       'Real Sociedad',
            'Barcelona',          'Celta Vigo',         'Real Madrid',
                'Eibar',             'Sevilla',            'Valencia',
         

In [27]:
soccer_data_all_teams = (
    matches_interim
        .groupby('season')['h_title']
            .unique()
            .to_dict()
)

soccer_data_all_teams

{'2014/2015': <ArrowStringArray>
 [         'Málaga',         'Sevilla',         'Granada',         'Almería',
          'Levante',           'Eibar',      'Celta Vigo',       'Barcelona',
   'Rayo Vallecano',     'Real Madrid',        'Valencia',          'Getafe',
         'Espanyol',         'Córdoba',   'Athletic Club', 'Atlético Madrid',
    'Dep La Coruña',      'Villarreal',           'Elche',   'Real Sociedad']
 Length: 20, dtype: str,
 '2015/2016': <ArrowStringArray>
 [         'Málaga',   'Dep La Coruña',        'Espanyol',  'Rayo Vallecano',
  'Atlético Madrid',   'Athletic Club',         'Levante',      'Real Betis',
   'Sporting Gijón',         'Granada',      'Villarreal',       'Barcelona',
    'Real Sociedad',      'Celta Vigo',     'Real Madrid',         'Sevilla',
         'Valencia',           'Eibar',          'Getafe',      'Las Palmas']
 Length: 20, dtype: str,
 '2016/2017': <ArrowStringArray>
 [         'Málaga',   'Dep La Coruña',         'Sevilla',         'Gra

Now as we have collected all of the teams names for each of the seasons for the two datasets, now lets proceed by using the `build_team_mapping` function which will automatically create a **dictionary** with the invalid names of the **Soccer data** teams and the corresponding valid **Understat teams** names!

For the function implemantation: [teams_names_mapper](../../src/football_betting_analysis/data/teams_names_mapper.py)

Before using the algorithm we should first create a **normalization function** which will remove any spaces and numeric symbols and replace them with single spaces!For this specific dataset some names are with accents such as: Almer**í**a.In order for the matching to work, the normalizations function should remove this accents and leave only valid unicode symbols!

I will use the `unicodedata` library which will split the word into symbols and accents.From the result I will only take the symbols!

In [28]:
def normalize(text: str) -> str:
    """Normalize text for matching"""
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    nfkd_form = unicodedata.normalize('NFKD', text)
    text = "".join([c for c in nfkd_form if not unicodedata.category(c) == 'Mn'])
    return text

There is one team name which is more specific and for it I should specifically provide the function with an exact alia.The function will first look if an aliases list is provided, if yes, it will look at him and directly put the two names into the dictionary!

In [29]:
EXACT_ALIASES = {
    "dep la coruna": "Deportivo La Coruna"
}

In [30]:
mapping, unresolved = build_team_mapping(
    normalize=normalize, 
    source_names=soccer_data_all_teams['2014/2015'], 
    target_names=understat_all_teams['2014/2015'],
    aliases=EXACT_ALIASES)

mapping

{'Málaga': 'Malaga',
 'Sevilla': 'Sevilla',
 'Granada': 'Granada',
 'Almería': 'Almeria',
 'Levante': 'Levante',
 'Eibar': 'Eibar',
 'Celta Vigo': 'Celta Vigo',
 'Barcelona': 'Barcelona',
 'Rayo Vallecano': 'Rayo Vallecano',
 'Real Madrid': 'Real Madrid',
 'Valencia': 'Valencia',
 'Getafe': 'Getafe',
 'Espanyol': 'Espanyol',
 'Córdoba': 'Cordoba',
 'Athletic Club': 'Athletic Club',
 'Atlético Madrid': 'Atletico Madrid',
 'Dep La Coruña': 'Deportivo La Coruna',
 'Villarreal': 'Villarreal',
 'Elche': 'Elche',
 'Real Sociedad': 'Real Sociedad'}

Now lets make some validations to ensure that everything is correct:

In [31]:
counter = 0
for key, value in mapping.items():
    if value in understat_all_teams['2014/2015']:
        counter += 1
        
counter

20

In [32]:
seasons = [f"{str(year)}/{str(year+1)}" for year in range(START_SEASON, END_SEASON)]

valid_teams = {}
for season in seasons:
    season = str(season)
    source_names = soccer_data_all_teams[season]
    target_names = understat_all_teams[season]
    
    # These are the results from the mapping algorithm
    mapping, unresolved = build_team_mapping(
        normalize=normalize, 
        source_names=source_names, 
        target_names=target_names,
        aliases=EXACT_ALIASES)
    
    counter = 0
    for key, value in mapping.items():
        if value in understat_all_teams[season]:
            counter += 1
            
    valid_teams[season] = counter

In [33]:
valid_teams

{'2014/2015': 20,
 '2015/2016': 20,
 '2016/2017': 20,
 '2017/2018': 20,
 '2018/2019': 20,
 '2019/2020': 20,
 '2020/2021': 20,
 '2021/2022': 20,
 '2022/2023': 20,
 '2023/2024': 20,
 '2024/2025': 20}

We can clearly see that there are 20 unique teams for each of the season with valid names!

## Applying the Mapping function:

So now as we have tested that algorithm that it is working, lets actually apply the algorithm to the dataset and rename all of the invalid La Liga teams names:

In [34]:
df = matches_interim.copy()

In [35]:
EXACT_ALIASES = {
    "dep la coruna": "Deportivo La Coruna"
}

In [36]:
seasons = [f"{str(year)}/{str(year+1)}" for year in range(START_SEASON, END_SEASON)]

season_mappings = {}
season_unresolved = {}

for season in seasons:
    source_names = soccer_data_all_teams[season]
    target_names = understat_all_teams[season]

    mapping, unresolved = build_team_mapping(
        normalize=normalize,
        source_names=source_names,
        target_names=target_names,
        aliases=EXACT_ALIASES
    )

    season_mappings[season] = mapping
    season_unresolved[season] = unresolved

Now the function which will use the mapping dicts that we have created from above and map the names row by row into the Football-Data dataset.

The function will accepts a `team_type_column` which is the column of the team name in the match: either the **HomeTeam** or the **AwayTeam**.The function should apply the mapping for the both dataset columns!

In [37]:
def map_team(row, team_type_column: str):
    season = row['season']
    team = row[team_type_column]

    mapping = season_mappings.get(season, {})

    return mapping.get(team, None)

Now lets finally apply the mapping into the dataset, using the function above.

First I will apply the function for the Home teams and then for the Away ones:

In [38]:
df['h_title_clean'] = df.apply(lambda row: map_team(row, 'h_title'), axis=1)

In [39]:
df['a_title_clean'] = df.apply(lambda row: map_team(row, 'a_title'), axis=1)

In [40]:
df[['h_title_clean', 'a_title_clean']]

,h_title_clean,a_title_clean
0,Malaga,Athletic Club
1,Sevilla,Valencia
2,Granada,Deportivo La Coruna
3,Almeria,Espanyol
4,Levante,Villarreal
...,...,...
4175,Alaves,Osasuna
4176,Leganes,Real Valladolid
4177,Girona,Atletico Madrid
4178,Villarreal,Sevilla


Now lets validated that everything has been ok:

In [41]:
df[
    (df['h_title_clean'].notna()) &
    (df['a_title_clean'].notna())
].shape

(4180, 18)

Now lets remove the helping columns and fill in the valid teams names into the original columns:

In [42]:
df['h_title'] = df['h_title_clean']
df['a_title'] = df['a_title_clean']

df = df.drop(columns='h_title_clean')
df = df.drop(columns='a_title_clean')

In [43]:
df.columns

Index(['league', 'season', 'date', 'time', 'h_title', 'a_title', 'h_goals',
       'a_goals', 'h_poss', 'a_poss', 'attendance', 'referee', 'h_formation',
       'a_formation', 'round', 'day'],
      dtype='str')

In [44]:
df.groupby('season')['h_title'].nunique()

season
2014/2015    20
2015/2016    20
2016/2017    20
2017/2018    20
2018/2019    20
2019/2020    20
2020/2021    20
2021/2022    20
2022/2023    20
2023/2024    20
2024/2025    20
Name: h_title, dtype: int64

In [45]:
df.groupby('season')['a_title'].nunique()

season
2014/2015    20
2015/2016    20
2016/2017    20
2017/2018    20
2018/2019    20
2019/2020    20
2020/2021    20
2021/2022    20
2022/2023    20
2023/2024    20
2024/2025    20
Name: a_title, dtype: int64

And now the most important validation: Are teams names in two datasets perfectly consistent?

In [46]:
valid_seasons = 0
for season in seasons:
    current_elo_matching_teams_names = sorted(df[df['season'] == season]['h_title'].unique())
    current_understat_teams_names = sorted(understat_matches[understat_matches['season'] == season]['h_title'].unique())
    
    if current_elo_matching_teams_names == current_understat_teams_names:
        valid_seasons += 1

valid_seasons

11

Everything is perfect!

In [47]:
matches_interim = df

In [48]:
matches_interim.info()

<class 'pandas.DataFrame'>
RangeIndex: 4180 entries, 0 to 4179
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   league       4180 non-null   str           
 1   season       4180 non-null   str           
 2   date         4180 non-null   datetime64[us]
 3   time         4180 non-null   str           
 4   h_title      4180 non-null   str           
 5   a_title      4180 non-null   str           
 6   h_goals      4180 non-null   int64         
 7   a_goals      4180 non-null   int64         
 8   h_poss       4179 non-null   Int64         
 9   a_poss       4179 non-null   Int64         
 10  attendance   3692 non-null   Int64         
 11  referee      4180 non-null   str           
 12  h_formation  3800 non-null   str           
 13  a_formation  3800 non-null   str           
 14  round        4180 non-null   str           
 15  day          4180 non-null   str           
dtypes: Int64(3), date

### Optimizing the dataset memory usage:
Using the `optimize_dataframe_memory` function which will try to reduce the memory usage of the dataset by parsing its features into more low-memory data types:

In [49]:
matches_interim = optimize_dataframe_memory(matches_interim)

Initial Memory Usage: 0.87 MB
Final Memory Usage: 0.19 MB
Memory Reduction: 78.08%

Optimized Data Types:
league               category
season               category
date           datetime64[us]
time                 category
h_title              category
a_title              category
h_goals                  int8
a_goals                  int8
h_poss                  Int64
a_poss                  Int64
attendance              Int64
referee              category
h_formation          category
a_formation          category
round                category
day                  category
dtype: object


### Final data validation:
As the dataset has been totally cleaned and many transformations and modifications has been applied on the data, it may be good if we made one final validation to ensure that everything is ready.

#### Validating the seasons matches counts:

In [50]:
matches_interim.groupby('season').size()

season
2014/2015    380
2015/2016    380
2016/2017    380
2017/2018    380
2018/2019    380
2019/2020    380
2020/2021    380
2021/2022    380
2022/2023    380
2023/2024    380
2024/2025    380
dtype: int64

### Validating the amount of unique teams in each of the seasons:

In [51]:
matches_interim.groupby('season')['h_title'].nunique()


season
2014/2015    20
2015/2016    20
2016/2017    20
2017/2018    20
2018/2019    20
2019/2020    20
2020/2021    20
2021/2022    20
2022/2023    20
2023/2024    20
2024/2025    20
Name: h_title, dtype: int64

In [52]:
matches_interim.groupby('season')['a_title'].nunique()

season
2014/2015    20
2015/2016    20
2016/2017    20
2017/2018    20
2018/2019    20
2019/2020    20
2020/2021    20
2021/2022    20
2022/2023    20
2023/2024    20
2024/2025    20
Name: a_title, dtype: int64

### Validating amount of matches for each of the teams in each of seasons:
Each team should have played a total of **38** matches per season.To check that, I will use the `validate_team_matches` function which will return the amount of seasons in which all of the teams has played a total of **38** matches!We expect to get **11** valid seasons: 

In [53]:
validate_team_matches(matches_interim, 'h_title', 'a_title')

{'2014/2015': 20,
 '2015/2016': 20,
 '2016/2017': 20,
 '2017/2018': 20,
 '2018/2019': 20,
 '2019/2020': 20,
 '2020/2021': 20,
 '2021/2022': 20,
 '2022/2023': 20,
 '2023/2024': 20,
 '2024/2025': 20}

### Checking for duplicates:

In [54]:
matches_interim.duplicated(
    subset=[
       'date', 'h_title', 'a_title' 
    ]
).any()

np.False_

Everything seems to be perfect!

### Saving the dataset:
After the dataset has been cleaned, I will save it into the `data/interim` folder.To do so, I will use the `save_data` function.

I will first need to create the **interim** folder of the Soccerdata dataset:

In [55]:
PROJECT_ROOT = Path().resolve().parent.parent

file_path = PROJECT_ROOT / SOCCER_DATA_INTERIM_PATH
file_path.mkdir(parents=True, exist_ok=True)

In [56]:
PROJECT_ROOT = Path().resolve().parent.parent
file_path = PROJECT_ROOT / SOCCER_DATA_INTERIM_PATH / 'interim_matches_v1.parquet'

save_data(data=matches_interim, file_path=file_path)

The file has already been created and it contains the exact data as the original dataset!
